# AIC2026 — ma hoa anh bang model thu hai (ViT-gopt-16-SigLIP2-384)

Huong dan day du: `docs/17_kaggle_encode_va_caption.md` trong repo.

**Settings truoc khi chay**
- Accelerator = **P100**. `08_encode.py` chay tren MOT gpu (`cuda:0`); chon
  T4 x2 la de mot nua phan cung nam khong ma van tinh du quota.
- Internet = **ON**
- Input = `aic2026-index` + 9 dataset anh (da gan san)

## 1. Ma nguon

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -q -b giai-doan-0 https://github.com/QuocKhanhDev-it/AIC_2026_FirstDance.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip -q install open_clip_torch pandas pyarrow

## 2. Chep `master.parquet` tu Private Dataset

TIM file chu KHONG doan duong dan: ten thu muc mount trong `/kaggle/input`
khong phai luc nao cung bang slug dataset. Doan cung thi `cp` bao
*No such file or directory*, roi moi cell sau chet day chuyen.

In [ ]:
import glob, shutil, os, pathlib
print("co trong /kaggle/input:", os.listdir('/kaggle/input'))
hit = glob.glob('/kaggle/input/**/master.parquet', recursive=True)
assert hit, "khong thay master.parquet — da Add Input dataset aic2026-index chua?"
pathlib.Path('index').mkdir(exist_ok=True)
shutil.copy(hit[0], 'index/master.parquet')
print("chep tu", hit[0])

## 3. Va duong dan — BUOC BAT BUOC, va la buoc de quen nhat

`kf_path` trong `master.parquet` la duong dan tuyet doi cua may dung index
(`D:\Project\...`). Tren Kaggle no khong ton tai. Bo qua buoc nay thi
`08_encode.py` thay **khong co anh nao** va ghi ra mot ma tran toan so 0 —
**khong bao loi gi**.

In [ ]:
!python scripts/12_va_duong_dan.py --roots /kaggle/input --ghi

### Chot chan — sai so nay thi DUNG LAI, dung encode

In [ ]:
import pandas as pd
m = pd.read_parquet('index/master.parquet')
n = m.kf_path.notna().sum()
print(f"co anh: {n:,} / {len(m):,}")
print(m[m.kf_path.notna()].video_id.str[:3].value_counts().sort_index())
assert n > 90_000, "duong dan chua va duoc — dataset anh chua mount xong?"

## 4. Bac 0 — 100 video, chi de soat

Doc 4 thu trong log truoc khi leo tiep:

| doc gi | phai la |
| --- | --- |
| dong `GPU:` | `Tesla P100` / `T4` — ra `cpu` la cham gap ~50 lan |
| so chieu | **1536** — ra 1152 la dang chay nham model cu |
| `--kiem-lech-hang` | dat |
| **anh/giay** | ghi lai, ca ke hoach dua vao con so nay |

`--workers 4` vi Kaggle cap 4 vCPU; dat 8 la cac tien trinh doc anh gianh nhau.
Tran VRAM thi ha `--batch` xuong 16.

In [ ]:
!python scripts/08_encode.py --model ViT-gopt-16-SigLIP2-384 --pretrained webli \
    --videos 100 --workers 4 --batch 32 --out /kaggle/working/thu.npy

Lech hang la loi nguy hiem nhat o day: moi vector ve sai anh, cosine van dep,
moi kiem tra cau truc van xanh, va diem thi tut ma khong ai biet vi sao.

In [ ]:
!python scripts/08_encode.py --kiem-lech-hang /kaggle/working/thu.npy

### Gio thuc te cho phan con lai

Chua co so do cho model nay tren phan cung Kaggle. No ~1,1 ty tham so, gap gan
ba lan SO400M — dung suy ra tu con so 23,9 anh/giay do tren may khac, gpu khac.

Ra duoi ~4 anh/giay thi tong vuot 7 gio. Luc do doi sang
`ViT-L-16-SigLIP2-384` (1024 chieu, nhe hon nhieu): van la model thu hai doc
lap, va mot ma tran chay xong dang gia hon mot ma tran chay do.

In [ ]:
ANH_MOI_GIAY = 0      # <-- dien so THAT doc duoc o bac 0
if ANH_MOI_GIAY:
    print(f"97.731 anh con lai -> {97731 / ANH_MOI_GIAY / 3600:.1f} gio")
    print("moi phien Kaggle toi da 12 gio, quota 30 gio/tuan")

## 5. Bac 1-5 — encode theo nhom roi ghep

| bac | pham vi | anh | cong don |
| ---: | --- | ---: | ---: |
| 1 | L23 | 2.326 | 2.326 |
| 2 | L27, L24 | 11.695 | 14.021 |
| 3 | L21, L30, L22 | 24.811 | 38.832 |
| 4 | L28, L29 | 21.454 | 60.286 |
| 5 | L25 | 37.445 | **97.731** |

Doi `BAC_DANG_LAM` roi chay hai cell duoi. Moi bac la mot vong tron ven:
encode -> tai ve -> ghep o may local -> do.

In [ ]:
BAC = {1: ['L23'], 2: ['L27', 'L24'], 3: ['L21', 'L30', 'L22'],
       4: ['L28', 'L29'], 5: ['L25']}
BAC_DANG_LAM = 1

import pandas as pd
m = pd.read_parquet('index/master.parquet')
nhom = BAC[BAC_DANG_LAM]
v = sorted(m[m.video_id.str[:3].isin(nhom) & m.kf_path.notna()].video_id.unique())
TEN_DS = f'ds_bac{BAC_DANG_LAM}.txt'
RA = f'/kaggle/working/clip_gopt_bac{BAC_DANG_LAM}.npy'
open(TEN_DS, 'w').write('\n'.join(v) + '\n')
so_anh = int(m[m.video_id.isin(v)].kf_path.notna().sum())
print(f"bac {BAC_DANG_LAM}: {nhom} -> {len(v)} video, {so_anh:,} anh")

In [ ]:
!python scripts/08_encode.py --model ViT-gopt-16-SigLIP2-384 --pretrained webli \
    --chi-video $TEN_DS --workers 4 --batch 32 --out $RA

## 6. Cache truy van — PHAI sinh, va sinh o DAY

`index/truy_van.npz` hien tai ma hoa bang thap van ban SO400M, **1152 chieu**.
Model nay **1536 chieu** — khong co cach nao dung lai cache cu. Cho nay khong
hong im lang (`KenhAnhCache` so so chieu roi dung han), nhung dung de no bat.

Sinh trong **cung notebook nay** vi model ~4 GB da tai san o day. Tach ra
notebook khac la tai lai lan nua, ton quota cho mot viec chi mat vai phut.

`--matrix` la du: script doc ten model va so chieu tu sidecar `.json` ma
`08_encode.py` vua ghi.

> Muon ma hoa ca de thi thi them `--de de_p2` va Add Input mot Private Dataset
> chua de — repo cong khai khong mang de theo.

In [ ]:
import glob, shutil, os
sc = sorted(glob.glob('/kaggle/working/clip_gopt_bac*.npy'))
print("ma tran da co:", [os.path.basename(x) for x in sc])
# sidecar phai nam CANH ma tran trong index/ thi 25_ moi doc duoc ten model
for f in sc + [x[:-4] + '.json' for x in sc]:
    if os.path.exists(f):
        shutil.copy(f, 'index/' + os.path.basename(f))

In [ ]:
TEN_MA_TRAN = f'clip_gopt_bac{BAC_DANG_LAM}.npy'
!python scripts/25_ma_hoa_truy_van.py --matrix $TEN_MA_TRAN --ra index/truy_van_gopt.npz --tap-dev --fp16
!cp index/truy_van_gopt.npz /kaggle/working/

## 7. Soat truoc khi tai ve

Tai `clip_gopt_bac*.npy` + file `.json` cung ten, va `truy_van_gopt.npz`.

> ⚠️ **KHONG tai `master.parquet` tu Kaggle ve.** File do da bi cell 3 va thanh
> duong dan `/kaggle/input/...`. De no len may local la moi thu doc anh chet
> hang loat. Chi tai `.npy`, `.json`, `.npz`.

In [ ]:
import numpy as np, glob, os
for f in sorted(glob.glob('/kaggle/working/clip_gopt_bac*.npy')):
    a = np.load(f, mmap_mode='r')
    print(f"{os.path.basename(f):26} {str(a.shape):18} {a.dtype}  "
          f"co vector: {int((np.abs(a).sum(1) > 0).sum()):,}")
!ls -la /kaggle/working/